# DQMBot Batch Query
Loads API key from `.env`, sends a list of prompts to the FNAL OpenWebUI instance, and collects results into a DataFrame.

In [ ]:
# --- Install dotenv if needed (already present on EAF, but just in case) ---
# !pip install python-dotenv --quiet

In [ ]:
import sys
import time
import pandas as pd
from pathlib import Path
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))
from owui_client import list_models, query

print('owui_client loaded OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL        = ''                              # leave '' to use OWUI_MODEL env var
DELAY        = 1.0                             # seconds between requests
CSV_PATH     = Path('document_chunks.csv')     # local archi RAG knowledge base
RAG_BACKEND  = 'local'                         # 'local' or 'owui'
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
print('=== Models ===')
for m in list_models():
    print(' ', m)

In [ ]:
# ── Prompts ───────────────────────────────────────────────────────────────────
# Edit this list with your actual DQM questions
SYSTEM_PROMPT = (
    'You are a CMS DQM expert assistant. '
    'Answer concisely and technically. '
    'If you are unsure, say so.'
)

PROMPTS = [
    'What does a hot strip in the SiStrip occupancy map indicate?',
    'Describe the typical signature of a ECAL supercrystal with a stuck ADC.',
    'What DQM alarm should fire when the CSC local trigger efficiency drops below 95%?',
    'How do you distinguish a noisy channel from a dead channel in the pixel detector DQM?',
    'What is the expected number of primary vertices per event at Run 3 pileup conditions?',
]
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Run batch ─────────────────────────────────────────────────────────────────
results = []
for i, prompt in enumerate(PROMPTS):
    print(f'[{i+1}/{len(PROMPTS)}] Querying...', end=' ', flush=True)
    result = query(
        prompt,
        model=MODEL,
        system=SYSTEM_PROMPT,
        rag_backend=RAG_BACKEND,
        csv_path=CSV_PATH,
    )
    results.append(result)
    status = 'ERROR' if result['error'] else f'{result["latency_s"]}s'
    print(status)
    time.sleep(DELAY)

print('\nDone.')

In [ ]:
# ── Review results ────────────────────────────────────────────────────────────
df = pd.DataFrame(results)

# Wide display so responses aren't truncated
pd.set_option('display.max_colwidth', 300)
display(df[['prompt', 'response', 'latency_s', 'error']])

In [ ]:
# ── Print responses readably ──────────────────────────────────────────────────
for row in results:
    print('='*72)
    print(f'PROMPT:  {row["prompt"]}')
    print(f'LATENCY: {row["latency_s"]}s  |  MODEL: {row["model_used"]}')
    print()
    if row['error']:
        print(f'ERROR: {row["error"]}')
    else:
        print(row['response'])
    print()

In [ ]:
# ── Save to CSV (optional) ────────────────────────────────────────────────────
out_path = 'dqmbot_results.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')